# Momepy roads 

This notebook uses momepy to calculate geometrical features of the streets, such as width and openess.

## Import libraries

In [8]:
import momepy as mm
import geopandas as gpd
import osmnx as ox
import pandas as pd

## Import noise segments

In [2]:
noise_streets = gpd.read_file("../../layers/BCN_noise_streets.gpkg")
print(noise_streets.crs)  #CRS = Coordinate Reference System
print(noise_streets.shape)
print(noise_streets.columns.tolist())
print("Number of street segments:", len(noise_streets))

EPSG:25831
(15115, 30)
['TRAM', 'TOTAL_D', 'TOTAL_E', 'TOTAL_N', 'TOTAL_DEN', 'TRANSIT_D', 'TRANSIT_E', 'TRANSIT_N', 'TRANSIT_DEN', 'GI_TR_D', 'GI_TR_E', 'GI_TR_N', 'GI_TR_DEN', 'FFCC_D', 'FFCC_E', 'FFCC_N', 'FFCC_DEN', 'INDUST_D', 'INDUST_E', 'INDUST_N', 'INDUST_DEN', 'VIANANTS_D', 'VIANANTS_E', 'OCI_N', 'PATIS_D', 'PATIS_E', 'geometry_type', 'start', 'end', 'geometry']
Number of street segments: 15115


## Import and project buildings

In [3]:
place_name = "Barcelona, Spain"
tags = {'building': True}

print("Downloading buildings from OSM...")
try:
    buildings = ox.features_from_place(place_name, tags)
except AttributeError:
    buildings = ox.geometries_from_place(place_name, tags)
    
# Project buildings to the same CRS as noise_streets
buildings = buildings.to_crs(noise_streets.crs)

## Create Momepy street profile

In [4]:
building_buffer = buildings.buffer(5).union_all()
profile = mm.street_profile(noise_streets, buildings)
profile.head(10)

,width,openness,width_deviation
0,37.346598,0.666667,0.865625
1,36.609682,0.700000,1.447206
2,33.535044,0.700000,2.498922
3,6.962798,0.388889,2.221544
4,19.754521,0.708333,0.564114
5,15.404320,0.454545,5.060507
6,9.382715,0.200000,0.530458
7,6.549761,0.285714,0.495146
8,36.722212,0.710526,4.759458
9,29.814184,0.687500,0.021472


## Add geo features to noise streets

In [9]:
dataset = pd.DataFrame({
    "street_id": noise_streets['TRAM'],
})
dataset['road_width'] = profile['width']
dataset['openness'] = profile['openness']
dataset.head(10)

,street_id,road_width,openness
0,T04719W,37.346598,0.666667
1,T19941Z,36.609682,0.700000
2,T18111R,33.535044,0.700000
3,T03222Y,6.962798,0.388889
4,T17625I,19.754521,0.708333
5,T05360P,15.404320,0.454545
6,T08863T,9.382715,0.200000
7,T00236S,6.549761,0.285714
8,T13009A,36.722212,0.710526
9,T11921P,29.814184,0.687500


## Export dataset to CSV

In [10]:
import os
output_dir = "../../data/processed"
os.makedirs(output_dir, exist_ok=True)
dataset.to_csv(os.path.join(output_dir, "momepy_streets.csv"), index=False)
print("Exported momepy_streets.csv")

Exported momepy_streets.csv


TODO: using building height we could also calculate street canyon height and h/w ratio, which are relevant for noise propagation.